<!-- NB_DOC_INTRO_V1 -->
# Maintenance predictive — NASA C-MAPSS Turbofan (RUL prediction)

> 📚 **Doc thematique** : [docs/06_TS_TDS.md](docs/06_TS_TDS.md) (Time Series & Traitement du Signal)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Cas industriel reference** : NASA C-MAPSS Turbofan Engine Degradation Simulation. Predire le **Remaining Useful Life (RUL)** = nb de cycles restants avant panne. 100+ moteurs, 21 capteurs, time series multivariees.

Ce notebook execute une **version simplifiee reproductible** (simulation des trajectoires de degradation) avec feature engineering + GradientBoosting Regressor.

Pour le vrai dataset : https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/

## Plan

1. Setup + simulation C-MAPSS-like
2. Visualisation degradation par engine
3. Feature engineering (rolling, polynomial fit)
4. Train/test split par engine_id
5. GradientBoosting Regressor pour RUL
6. Eval (MAE, RMSE, NASA scoring function)
7. Pieges + References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Simulation C-MAPSS-like

In [ ]:
# 50 moteurs, chacun tourne pendant 100-300 cycles puis 'tombe en panne'
# 3 capteurs simules : temp_oil, vibration, pressure

n_engines = 50
data = []
for engine_id in range(n_engines):
    n_cycles = np.random.randint(150, 350)
    # Degradation lineaire + bruit + saut a la fin
    t = np.arange(n_cycles)
    temp = 50 + 0.1 * t + 0.0001 * t**2 + np.random.normal(0, 2, n_cycles)
    vib = 0.5 + 0.005 * t + np.random.normal(0, 0.1, n_cycles)
    press = 100 - 0.05 * t + np.random.normal(0, 1, n_cycles)
    rul = n_cycles - 1 - t      # cycles restants jusqu'a panne
    for c in range(n_cycles):
        data.append({"engine_id": engine_id, "cycle": c,
                     "temp": temp[c], "vibration": vib[c], "pressure": press[c],
                     "RUL": rul[c]})

df = pd.DataFrame(data)
print(f"Total samples : {len(df)}, engines : {df['engine_id'].nunique()}")
df.head()

## 2. Visualisation degradation

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8))
for engine_id in range(5):
    sub = df[df["engine_id"] == engine_id]
    axes[0].plot(sub["cycle"], sub["temp"], alpha=0.6, label=f"engine {engine_id}")
    axes[1].plot(sub["cycle"], sub["vibration"], alpha=0.6)
    axes[2].plot(sub["cycle"], sub["pressure"], alpha=0.6)
axes[0].set_title("Temp"); axes[0].legend(loc="best", fontsize=7)
axes[1].set_title("Vibration")
axes[2].set_title("Pressure"); axes[2].set_xlabel("Cycle")
plt.tight_layout(); plt.show()

## 3. Feature engineering avance — rolling par engine

In [ ]:
def add_rolling_features(df, sensors, windows=[5, 20]):
    out = df.copy()
    for s in sensors:
        for w in windows:
            out[f"{s}_roll_mean_{w}"] = df.groupby("engine_id")[s].transform(lambda x: x.rolling(w, min_periods=1).mean())
            out[f"{s}_roll_std_{w}"] = df.groupby("engine_id")[s].transform(lambda x: x.rolling(w, min_periods=1).std())
        out[f"{s}_diff"] = df.groupby("engine_id")[s].diff().fillna(0)
    return out

sensors = ["temp", "vibration", "pressure"]
df_feat = add_rolling_features(df, sensors)
print(f"Features shape : {df_feat.shape}")
df_feat.head()

## 4. Train/test split par engine_id (anti-leakage)

In [ ]:
# Crucial : split par engine, pas par sample !
np.random.seed(SEED)
engines = df["engine_id"].unique()
np.random.shuffle(engines)
split = int(len(engines) * 0.75)
train_engines = engines[:split]
test_engines = engines[split:]

train = df_feat[df_feat["engine_id"].isin(train_engines)]
test = df_feat[df_feat["engine_id"].isin(test_engines)]
print(f"Train : {len(train)} samples, {len(train_engines)} engines")
print(f"Test  : {len(test)} samples,  {len(test_engines)} engines")

feature_cols = [c for c in df_feat.columns if c not in ["engine_id", "cycle", "RUL"]]
X_tr, y_tr = train[feature_cols].fillna(0), train["RUL"]
X_te, y_te = test[feature_cols].fillna(0), test["RUL"]

## 5. GradientBoosting Regressor

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                                  random_state=SEED).fit(X_tr, y_tr)
pred = gbr.predict(X_te)
print(f"MAE  : {mean_absolute_error(y_te, pred):.2f} cycles")
print(f"RMSE : {root_mean_squared_error(y_te, pred):.2f} cycles")
print(f"RUL test range : [{y_te.min()}, {y_te.max()}]")

## 6. NASA scoring function (asymetrique : pénalise les sous-estimations)

In [ ]:
# NASA score : exp(-d/13) - 1 si d < 0 (under-predict, dangereux !)
#              exp(d/10) - 1 si d >= 0 (over-predict, moins grave)
def nasa_score(y_true, y_pred):
    d = y_pred - y_true
    return np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1).sum()

print(f"NASA score (plus bas = mieux) : {nasa_score(y_te.values, pred):.0f}")
print(f"(Baseline naif RUL=moyenne : {nasa_score(y_te.values, np.full_like(y_te.values, y_tr.mean()).astype(float)):.0f})")

## 7. Visualisation predictions vs ground truth

In [ ]:
# Pour 3 engines de test
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, eid in enumerate(test_engines[:3]):
    sub_te = test[test["engine_id"] == eid].sort_values("cycle")
    pred_sub = gbr.predict(sub_te[feature_cols].fillna(0))
    axes[i].plot(sub_te["cycle"], sub_te["RUL"], label="True RUL", color="black")
    axes[i].plot(sub_te["cycle"], pred_sub, label="Predicted", color="red", alpha=0.7)
    axes[i].set_title(f"Engine {eid}"); axes[i].set_xlabel("Cycle"); axes[i].set_ylabel("RUL")
    axes[i].legend()
plt.tight_layout(); plt.show()

## 8. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Split par sample (pas par engine) | LEAK : meme engine train + test ! |
| Predire RUL absolue sans clip | Souvent on clip RUL a 130 (early cycles non utiles) |
| Reporter MAE seule | NASA score asymetrique (sous-estimer = danger) |
| Pas regarder par engine_id | Performance globale masque les engines difficiles |
| Pas augmenter avec features rolling | Capture la tendance de degradation |
| Predire RUL sans contexte historique | Lag features / RNN/LSTM utiles |
| Ignorer survival analysis | Cf. [DS_Survival_Analysis.ipynb](./DS_Survival_Analysis.ipynb) |

## 9. Modeles modernes pour cette tache

| Approche | Forces |
|---|---|
| **GradientBoosting / LightGBM** (ici) | Robuste, peu de tuning |
| **LSTM / GRU** sur sequences capteurs | Capture dynamique temporelle |
| **TFT** (Temporal Fusion Transformer) | SOTA TS multivariee |
| **N-BEATS** | Bench-winning sur compet RUL |
| **Survival Analysis** (DeepSurv) | Modele censure naturel |


## References

### Documentation
- NASA C-MAPSS dataset : https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/
- Kaggle mirror : https://www.kaggle.com/datasets/behrad3d/nasa-cmaps
- PyTorch Forecasting (TFT) : https://pytorch-forecasting.readthedocs.io/
- darts (multi-model TS) : https://unit8co.github.io/darts/

### Voir aussi
- [TS_Maintenance_Prédictive.ipynb](TS_Maintenance_Prédictive.ipynb)
- [TS_Time_Series_Overview.ipynb](TS_Time_Series_Overview.ipynb)
- [DS_Survival_Analysis.ipynb](DS_Survival_Analysis.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
